# Benchmark Scenario 002 — Cố định Framework, So sánh Backbone

**KB2**: Framework = CSQ (tốt nhất từ KB1). So sánh 4 backbone: AlexNet, ResNet, ViT-B_32, ViT-B_16 trên CIFAR-10 / bit=32 / epoch=150.

## Workflow
1. Chạy **Cell 1–4** (mount Drive → clone repo → cài deps → symlink) mỗi khi mở Colab runtime mới.
2. Chạy **Cell 5** (config) — đổi `EPOCH=10` cho smoke test local, `EPOCH=150` cho Colab.
3. Chạy **Cell 6** (smoke test) — kiểm tra 4 backbone × epoch 10 không crash trước khi tốn GPU.
4. Mở **4 runtime Colab song song**, mỗi runtime chạy 1 trong Cell 7–10 để rút ngắn wall-clock.

Kết quả ghi vào `Checkpoints_Results/CSQ-{backbone}-cifar10/` (persist qua Drive symlink).

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
# Cell 2 — Clone repo và chuyển vào thư mục dự án
!git clone https://github.com/hatuan314/VisionTransformerHashing.git
%cd VisionTransformerHashing

In [ ]:
# Cell 3 — Cài dependency thiếu trên Colab
!pip install ml_collections

In [ ]:
# Cell 4 — Tạo symlink tới Drive (pretrainedVIT, dataset, Checkpoints_Results)
import os

DRIVE_BASE = '/content/gdrive/MyDrive/master_is/semester_3/IR/VTS-LAB'

for link in ['pretrainedVIT', 'dataset', 'Checkpoints_Results']:
    if os.path.islink(link):
        os.remove(link)

os.symlink(f'{DRIVE_BASE}/pretrainedVIT', 'pretrainedVIT')
os.symlink(f'{DRIVE_BASE}/dataset', 'dataset')

os.makedirs(f'{DRIVE_BASE}/Checkpoints_Results', exist_ok=True)
os.symlink(f'{DRIVE_BASE}/Checkpoints_Results', 'Checkpoints_Results')

!ls pretrainedVIT/
!ls dataset/
!ls Checkpoints_Results/

In [ ]:
# Cell 5 — Config
EPOCH = 150        # đổi xuống 10 cho smoke test local
TEST_MAP = 30      # đổi xuống 5 cho smoke
BIT = 32
FRAMEWORK = "CSQ"
DATASET = "cifar10"
BACKBONES = ["AlexNet", "ResNet", "ViT-B_32", "ViT-B_16"]

def save_path_for(bb):
    return f"Checkpoints_Results/{FRAMEWORK}-{bb}-{DATASET}"

print(f"EPOCH={EPOCH} | BIT={BIT} | DATASET={DATASET} | FRAMEWORK={FRAMEWORK}")
print("Backbones:", BACKBONES)

In [ ]:
# Cell 6 — Smoke test: 4 backbone × epoch 10 tuần tự
import subprocess

SMOKE_EPOCH, SMOKE_TEST_MAP = 10, 5
results = {}
for bb in BACKBONES:
    cmd = [
        "python", f"{FRAMEWORK}.py",
        "--dataset", DATASET,
        "--bit", str(BIT),
        "--epoch", str(SMOKE_EPOCH),
        "--test_map", str(SMOKE_TEST_MAP),
        "--backbone", bb,
        "--save_path", save_path_for(bb),
    ]
    rc = subprocess.call(cmd)
    results[bb] = "OK" if rc == 0 else f"FAIL(rc={rc})"

print("\n=== Smoke Test Results ===")
for bb, st in results.items():
    print(f"  {bb:12s} {st}")

In [ ]:
# Cell 7 — Train AlexNet (chạy trên 1 Colab runtime riêng)
!python CSQ.py --dataset {DATASET} --bit {BIT} --epoch {EPOCH} --test_map {TEST_MAP} --backbone AlexNet --save_path {save_path_for("AlexNet")}

In [ ]:
# Cell 8 — Train ResNet (chạy trên 1 Colab runtime riêng)
!python CSQ.py --dataset {DATASET} --bit {BIT} --epoch {EPOCH} --test_map {TEST_MAP} --backbone ResNet --save_path {save_path_for("ResNet")}

In [ ]:
# Cell 9 — Train ViT-B_32 (chạy trên 1 Colab runtime riêng)
!python CSQ.py --dataset {DATASET} --bit {BIT} --epoch {EPOCH} --test_map {TEST_MAP} --backbone ViT-B_32 --save_path {save_path_for("ViT-B_32")}

In [ ]:
# Cell 10 — Train ViT-B_16 (chạy trên 1 Colab runtime riêng)
# Lưu ý: nếu OOM với batch_size=32, giảm batch_size xuống 16 trong get_config() rồi rerun
!python CSQ.py --dataset {DATASET} --bit {BIT} --epoch {EPOCH} --test_map {TEST_MAP} --backbone ViT-B_16 --save_path {save_path_for("ViT-B_16")}